In [1]:
#!/usr/bin/env python
# coding: utf-8

# =============================================================================
# 导入库，加载数据和CV设置
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import json
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# 创建输出文件夹
Path('models/rf').mkdir(parents=True, exist_ok=True)

# 加载数据
df = pd.read_csv('data/full_dataset.csv')

with open('data/cv_setup.pkl', 'rb') as f:
    cv_setup = pickle.load(f)
with open('data/cv_setup_inner.pkl', 'rb') as f:
    cv_setup_inner = pickle.load(f)

feature_cols = cv_setup['feature_cols']
target_col   = cv_setup['target_col']
outer_folds  = cv_setup['outer_folds']

X = df[feature_cols].values
y = df[target_col].values

# 超参数搜索空间
param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [10, 20, 30, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2]
}

# 边缘原子和原子类型配置
EDGE_THRESHOLD   = 0.1
ATOM_TYPE_COLS   = ['T_O', 'T_C', 'T_Ti1', 'T_Ti2']
ATOM_TYPE_LABELS = ['O', 'C', 'Ti_inner', 'Ti_outer']


# =============================================================================
# 辅助函数：计算指标，分类评估
# =============================================================================

def calculate_metrics(y_true, y_pred):
    return {
        'mae':       float(mean_absolute_error(y_true, y_pred)),
        'rmse':      float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'r2':        float(r2_score(y_true, y_pred)),
        'n_samples': int(len(y_true))
    }

def classify_and_evaluate(y_true, y_pred, df_subset):
    results = {}
    results['overall'] = calculate_metrics(y_true, y_pred)

    edge_mask     = df_subset['D_E'].values < EDGE_THRESHOLD
    interior_mask = ~edge_mask

    results['edge']     = calculate_metrics(y_true[edge_mask],     y_pred[edge_mask])     if edge_mask.sum()     > 0 else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}
    results['interior'] = calculate_metrics(y_true[interior_mask], y_pred[interior_mask]) if interior_mask.sum() > 0 else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}

    results['by_atom_type'] = {}
    for col, label in zip(ATOM_TYPE_COLS, ATOM_TYPE_LABELS):
        mask = df_subset[col].values == 1
        results['by_atom_type'][label] = calculate_metrics(y_true[mask], y_pred[mask]) if mask.sum() > 0 \
            else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}

    return results




In [2]:
# =============================================================================
# 嵌套交叉验证主循环
# =============================================================================

outer_results         = []
best_models           = []
all_train_predictions = []
all_test_predictions  = []
feature_importances   = []   # 每折的特征重要性
classification_results = {'fold_wise': {}, 'averaged': {}}

start_time = time.time()

for fold_idx, outer_fold in enumerate(outer_folds):
    train_idx = outer_fold['train_atoms_idx']
    test_idx  = outer_fold['test_atoms_idx']

    X_train = X[train_idx];  y_train = y[train_idx]
    X_test  = X[test_idx];   y_test  = y[test_idx]

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test  = df.iloc[test_idx].reset_index(drop=True)

    # 标准化（RF不严格要求，但保持一致性）
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # 构建内层CV索引
    inner_folds = cv_setup_inner[fold_idx]['inner_folds']
    inner_cv_indices = []
    for inner_fold in inner_folds:
        inner_train_set = set(inner_fold['inner_train_structures'])
        inner_val_set   = set(inner_fold['inner_val_structures'])
        struct_ids = df.iloc[train_idx]['structure_id'].values
        inner_cv_indices.append((
            np.where([s in inner_train_set for s in struct_ids])[0],
            np.where([s in inner_val_set   for s in struct_ids])[0]
        ))

    # 超参数搜索（RF内部已并行，GridSearch设n_jobs=1）
    grid_search = GridSearchCV(
        estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
        param_grid=param_grid,
        cv=inner_cv_indices,
        scoring='neg_mean_squared_error',
        n_jobs=1,
        verbose=0
    )
    grid_search.fit(X_train_scaled, y_train)
    best_params = grid_search.best_params_

    # 用最优参数训练
    best_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
    best_model.fit(X_train_scaled, y_train)

    # 收集特征重要性
    feature_importances.append({
        'fold': fold_idx + 1,
        'importances': best_model.feature_importances_.tolist()
    })

    y_train_pred = best_model.predict(X_train_scaled)
    y_test_pred  = best_model.predict(X_test_scaled)

    train_class_results = classify_and_evaluate(y_train, y_train_pred, df_train)
    test_class_results  = classify_and_evaluate(y_test,  y_test_pred,  df_test)

    print(f"折叠 {fold_idx+1} | 最优参数: {best_params} | "
          f"训练MAE: {train_class_results['overall']['mae']:.5f} | "
          f"测试MAE: {test_class_results['overall']['mae']:.5f}")

    fold_result = {
        'fold': fold_idx + 1,
        'best_params': best_params,
        'train_metrics': train_class_results,
        'test_metrics':  test_class_results,
        'n_train': len(train_idx),
        'n_test':  len(test_idx),
        'train_structures': outer_fold['train_structures'],
        'test_structures':  outer_fold['test_structures']
    }
    outer_results.append(fold_result)
    best_models.append({'fold': fold_idx + 1, 'model': best_model,
                        'scaler': scaler, 'best_params': best_params})

    classification_results['fold_wise'][f'fold_{fold_idx+1}'] = {
        'train': train_class_results,
        'test':  test_class_results
    }

    for i in range(len(train_idx)):
        all_train_predictions.append({
            'fold': fold_idx + 1,
            'atom_index': int(train_idx[i]),
            'structure_id': df_train.iloc[i]['structure_id'],
            'D_E': float(df_train.iloc[i]['D_E']),
            'T_O': int(df_train.iloc[i]['T_O']),
            'T_C': int(df_train.iloc[i]['T_C']),
            'T_Ti1': int(df_train.iloc[i]['T_Ti1']),
            'T_Ti2': int(df_train.iloc[i]['T_Ti2']),
            'y_true': float(y_train[i]),
            'y_pred': float(y_train_pred[i]),
            'error': float(y_train_pred[i] - y_train[i]),
            'abs_error': float(abs(y_train_pred[i] - y_train[i]))
        })
    for i in range(len(test_idx)):
        all_test_predictions.append({
            'fold': fold_idx + 1,
            'atom_index': int(test_idx[i]),
            'structure_id': df_test.iloc[i]['structure_id'],
            'D_E': float(df_test.iloc[i]['D_E']),
            'T_O': int(df_test.iloc[i]['T_O']),
            'T_C': int(df_test.iloc[i]['T_C']),
            'T_Ti1': int(df_test.iloc[i]['T_Ti1']),
            'T_Ti2': int(df_test.iloc[i]['T_Ti2']),
            'y_true': float(y_test[i]),
            'y_pred': float(y_test_pred[i]),
            'error': float(y_test_pred[i] - y_test[i]),
            'abs_error': float(abs(y_test_pred[i] - y_test[i]))
        })

elapsed_time = time.time() - start_time
print(f"\n嵌套CV完成，耗时 {elapsed_time:.1f} 秒 ({elapsed_time/60:.1f} 分钟)")




折叠 1 | 最优参数: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300} | 训练MAE: 0.00133 | 测试MAE: 0.00469
折叠 2 | 最优参数: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200} | 训练MAE: 0.00142 | 测试MAE: 0.00393
折叠 3 | 最优参数: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200} | 训练MAE: 0.00113 | 测试MAE: 0.00400
折叠 4 | 最优参数: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300} | 训练MAE: 0.00141 | 测试MAE: 0.00401
折叠 5 | 最优参数: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100} | 训练MAE: 0.00128 | 测试MAE: 0.00448

嵌套CV完成，耗时 1828.2 秒 (30.5 分钟)


In [3]:
# =============================================================================
# 计算平均性能，特征重要性，训练最终模型，保存所有结果
# =============================================================================

def calculate_averaged_metrics(outer_results, dataset_type='test'):
    key = f'{dataset_type}_metrics'
    averaged = {}

    for category in ['overall', 'edge', 'interior']:
        maes  = [r[key][category]['mae']  for r in outer_results]
        rmses = [r[key][category]['rmse'] for r in outer_results]
        r2s   = [r[key][category]['r2']   for r in outer_results]
        ns    = [r[key][category]['n_samples'] for r in outer_results]
        averaged[category] = {
            'mae_mean':  float(np.mean(maes)),  'mae_std':  float(np.std(maes)),
            'rmse_mean': float(np.mean(rmses)), 'rmse_std': float(np.std(rmses)),
            'r2_mean':   float(np.mean(r2s)),   'r2_std':   float(np.std(r2s)),
            'n_samples_mean': float(np.mean(ns))
        }

    averaged['by_atom_type'] = {}
    for label in ATOM_TYPE_LABELS:
        maes  = [r[key]['by_atom_type'][label]['mae']  for r in outer_results]
        rmses = [r[key]['by_atom_type'][label]['rmse'] for r in outer_results]
        r2s   = [r[key]['by_atom_type'][label]['r2']   for r in outer_results]
        ns    = [r[key]['by_atom_type'][label]['n_samples'] for r in outer_results]
        averaged['by_atom_type'][label] = {
            'mae_mean':  float(np.mean(maes)),  'mae_std':  float(np.std(maes)),
            'rmse_mean': float(np.mean(rmses)), 'rmse_std': float(np.std(rmses)),
            'r2_mean':   float(np.mean(r2s)),   'r2_std':   float(np.std(r2s)),
            'n_samples_mean': float(np.mean(ns))
        }

    return averaged

train_averaged = calculate_averaged_metrics(outer_results, 'train')
test_averaged  = calculate_averaged_metrics(outer_results, 'test')
classification_results['averaged'] = {'train': train_averaged, 'test': test_averaged}

# 打印平均性能
print("\n5折CV平均性能（测试集）:")
for cat in ['overall', 'edge', 'interior']:
    m = test_averaged[cat]
    print(f"  {cat:10s} - MAE: {m['mae_mean']:.5f} ± {m['mae_std']:.5f}  "
          f"RMSE: {m['rmse_mean']:.5f} ± {m['rmse_std']:.5f}  "
          f"R²: {m['r2_mean']:.6f} ± {m['r2_std']:.6f}")

# 特征重要性（5折平均）
mean_importances = np.mean([fi['importances'] for fi in feature_importances], axis=0)
std_importances  = np.std( [fi['importances'] for fi in feature_importances], axis=0)

importance_df = pd.DataFrame({
    'feature':    feature_cols,
    'importance': mean_importances,
    'std':        std_importances
}).sort_values('importance', ascending=False)

importance_df.to_csv('models/rf/rf_feature_importance.csv', index=False)
print("\nTop 10 特征重要性:")
print(importance_df.head(10).to_string(index=False))

# 最终模型（使用测试集MAE最小的折叠的超参数）
best_fold_idx = int(np.argmin([r['test_metrics']['overall']['mae'] for r in outer_results]))
final_params  = outer_results[best_fold_idx]['best_params']
print(f"\n最终模型参数（来自折叠 {best_fold_idx+1}）: {final_params}")

scaler_final   = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X)
final_model    = RandomForestRegressor(**final_params, random_state=42, n_jobs=-1)
final_model.fit(X_scaled_final, y)

# 保存预测CSV
df_train_pred_all = pd.DataFrame(all_train_predictions)
df_test_pred_all  = pd.DataFrame(all_test_predictions)

for fold_num in range(1, 6):
    df_train_pred_all[df_train_pred_all['fold'] == fold_num].to_csv(
        f'models/rf/fold_{fold_num}_train_predictions.csv', index=False)
    df_test_pred_all[df_test_pred_all['fold'] == fold_num].to_csv(
        f'models/rf/fold_{fold_num}_test_predictions.csv', index=False)

# 保存分类评分CSV
classification_rows = []
for fold_num in range(1, 6):
    fold_data = classification_results['fold_wise'][f'fold_{fold_num}']
    for dataset in ['train', 'test']:
        for category in ['overall', 'edge', 'interior']:
            d = fold_data[dataset][category]
            classification_rows.append({
                'fold': fold_num, 'dataset': dataset,
                'category': category, 'subcategory': 'all',
                'mae': d['mae'], 'rmse': d['rmse'],
                'r2': d['r2'], 'n_samples': d['n_samples']
            })
        for label in ATOM_TYPE_LABELS:
            d = fold_data[dataset]['by_atom_type'][label]
            classification_rows.append({
                'fold': fold_num, 'dataset': dataset,
                'category': 'atom_type', 'subcategory': label,
                'mae': d['mae'], 'rmse': d['rmse'],
                'r2': d['r2'], 'n_samples': d['n_samples']
            })

pd.DataFrame(classification_rows).to_csv(
    'models/rf/classification_scores.csv', index=False)

# 保存模型文件
model_package = {
    'model': final_model, 'scaler': scaler_final,
    'feature_cols': feature_cols, 'target_col': target_col,
    'best_params': final_params,
    'feature_importances': final_model.feature_importances_.tolist(),
    'cv_performance': {'train': train_averaged, 'test': test_averaged},
    'edge_threshold': EDGE_THRESHOLD,
    'atom_type_labels': ATOM_TYPE_LABELS
}
with open('models/rf/rf_final_model.pkl', 'wb') as f:
    pickle.dump(model_package, f)

with open('models/rf/rf_all_folds.pkl', 'wb') as f:
    pickle.dump(best_models, f)

results_dict = {
    'model_name': 'Random Forest',
    'n_folds': len(outer_folds),
    'n_features': len(feature_cols),
    'training_time_seconds': float(elapsed_time),
    'edge_threshold': EDGE_THRESHOLD,
    'param_grid': param_grid,
    'cv_results': {'train': train_averaged, 'test': test_averaged},
    'fold_details': outer_results,
    'feature_importances': {
        'features':        feature_cols,
        'mean_importance': mean_importances.tolist(),
        'std_importance':  std_importances.tolist(),
        'top_10':          importance_df.head(10).to_dict('records')
    },
    'final_model': {'params': final_params}
}
with open('models/rf/rf_results.json', 'w') as f:
    json.dump(results_dict, f, indent=4)

print("\n已保存: 预测CSV, classification_scores.csv, rf_feature_importance.csv, "
      "rf_final_model.pkl, rf_all_folds.pkl, rf_results.json")


5折CV平均性能（测试集）:
  overall    - MAE: 0.00422 ± 0.00031  RMSE: 0.00737 ± 0.00108  R²: 0.999915 ± 0.000027
  edge       - MAE: 0.00876 ± 0.00121  RMSE: 0.01593 ± 0.00389  R²: 0.999447 ± 0.000289
  interior   - MAE: 0.00369 ± 0.00022  RMSE: 0.00547 ± 0.00022  R²: 0.999955 ± 0.000004

Top 10 特征重要性:
feature  importance      std
     EN    0.499748 0.018694
  N_Ti2    0.477232 0.018710
    N_O    0.019989 0.000129
  N_Ti1    0.001015 0.000057
    N_C    0.000632 0.000028
    D_E    0.000435 0.000004
  T_Ti1    0.000237 0.000023
    Del    0.000213 0.000025
  M_Inf    0.000101 0.000008
    H_A    0.000070 0.000009

最终模型参数（来自折叠 2）: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}

已保存: 预测CSV, classification_scores.csv, rf_feature_importance.csv, rf_final_model.pkl, rf_all_folds.pkl, rf_results.json
